# Business Entity Resolution Pipeline — Kaggle Runner

This notebook runs the complete entity resolution pipeline in a Kaggle notebook environment.

### Recommended Kaggle Settings:
- **Accelerator:** GPU P100 or GPU T4 x2
- **Internet:** ON (for cloning repo, pip installing dependencies, and downloading HF embedding weights)
- **Persistence:** Files only (optional)

## 1. Environment & Hardware Verification

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Clone Repository & Install Dependencies

In [ ]:
import os

# Work inside /kaggle/working
%cd /kaggle/working

# Clone the repo if not already cloned
if not os.path.exists("amazon-ml"):
    !git clone https://github.com/tamcee/amazon-ml.git

%cd /kaggle/working/amazon-ml/student_resource/code/business_entity_resolution
!git pull origin main

# Install dependencies
!pip install -q -r requirements.txt

## 3. Dataset Configuration

On Kaggle, datasets attached to the notebook appear under `/kaggle/input/<dataset-name>/`.

This cell searches for the dataset files or allows you to specify the directory.

In [ ]:
import os
import glob

# Auto-detect dataset directory under /kaggle/input
input_root = "/kaggle/input"
train_dir = None
test_dir = None

# Search for train_source1.tsv and test_source1.tsv
for root, dirs, files in os.walk(input_root):
    if "train_source1.tsv" in files:
        train_dir = root
    if "test_source1.tsv" in files:
        test_dir = root

# Fallback check if dataset exists in repo structure
if not train_dir and os.path.exists("../../dataset/train"):
    train_dir = os.path.abspath("../../dataset/train")
if not test_dir and os.path.exists("../../dataset/test"):
    test_dir = os.path.abspath("../../dataset/test")

# Output directory in Kaggle working area
output_dir = "/kaggle/working/output"
os.makedirs(output_dir, exist_ok=True)

# Export environment variables for the pipeline
if train_dir:
    os.environ['TRAIN_DIR'] = train_dir
    print(f"Detected TRAIN_DIR: {train_dir}")
else:
    print("WARNING: train_source1.tsv not found in /kaggle/input. Please set os.environ['TRAIN_DIR'] manually.")

if test_dir:
    os.environ['TEST_DIR'] = test_dir
    print(f"Detected TEST_DIR:  {test_dir}")
else:
    print("WARNING: test_source1.tsv not found in /kaggle/input. Please set os.environ['TEST_DIR'] manually.")

os.environ['OUTPUT_DIR'] = output_dir
print(f"Set OUTPUT_DIR:    {output_dir}")

## 4. Optional: Quick Smoke Test with Dev Cohort

Run on a 5,000-entity cohort first to confirm blocking recall and pipeline health before full scale.

In [ ]:
# Run dev cohort creation & quick training (set DEV_TEST=True to run, False to skip)
DEV_TEST = False

if DEV_TEST:
    !python run_pipeline.py --stage dev_cohort --dev-size 5000
    !TRAIN_DIR=artifacts/dev_cohort python run_pipeline.py --stage train

## 5. Full Pipeline: Train & Test Inference

In [ ]:
# Stage 1: Full-scale training
!python run_pipeline.py --stage train

In [ ]:
# Stage 2: Full-scale test inference
!python run_pipeline.py --stage infer

## 6. Official Validation Check

In [ ]:
validator_path = "../../utils/validate_submission.py"
matching_file = os.path.join(os.environ['OUTPUT_DIR'], "matching_results.tsv")
candidate_file = os.path.join(os.environ['OUTPUT_DIR'], "candidate_pairs.tsv")
test_dir_path = os.environ['TEST_DIR']

!python {validator_path} \
    --matching {matching_file} \
    --candidate {candidate_file} \
    --test-dir {test_dir_path}

## 7. Package Final Submissions for Download

In Kaggle, any files in `/kaggle/working/` appear in the notebook output tab for download.

In [ ]:
!ls -lh /kaggle/working/output/
!zip -j /kaggle/working/submission_results.zip /kaggle/working/output/matching_results.tsv /kaggle/working/output/candidate_pairs.tsv
print("Submission package created at /kaggle/working/submission_results.zip")